# **Sistema Híbrido de Predicción y Gestión de Riesgo (FNN + GARCH)**

### Segundo examen parcial

https://es.finance.yahoo.com/quote/RB%3DF/

+ Clara Aguilar 

+ Samantha Sanchez 

+ Juan Pablo Colome

+ Isabel Valladolid

-----------------------------------------------------------------------------------------------

En este examen se desarrolla un sistema híbrido para la predicción y gestión de riesgo utilizando como activo la gasolina RBOB. Se implementa una red neuronal feedforward (FFNN) para estimar los precios de cierre en el corto plazo, buscando capturar la dirección del mercado, mientras que un modelo GARCH permite modelar la volatilidad y calcular el Value at Risk (VaR) al 95%. 

Este enfoque conjunto permite no solo anticipar movimientos del precio, sino también cuantificar el riesgo asociado, aportando una base más sólida para la toma de decisiones financieras.

In [25]:
%%capture
# !pip install  scikit-learn tensorflow
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import acf, pacf
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Obtención y limpieza de datos

Se descargaron datos históricos diarios de la gasolina RBOB (RB=F) utilizando yfinance, cubriendo los últimos 10 años. Esto permite contar con suficiente información para el análisis y modelado. Finalmente, se visualizan las primeras filas para verificar la estructura del dataset.

In [5]:
# Descargar datos
df = yf.download("RB=F", period="10y", interval="1d")

df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,RB=F,RB=F,RB=F,RB=F,RB=F
Date,,,,,
2016-04-12,1.5343,1.5386,1.4932,1.5023,58656
2016-04-13,1.5295,1.5384,1.5059,1.5277,62404
2016-04-14,1.5056,1.5425,1.4957,1.5273,46239
2016-04-15,1.4612,1.5126,1.4534,1.5064,50578
2016-04-18,1.4365,1.4617,1.3990,1.4229,49114


Se estandarizan los nombres de las columnas a minúsculas para facilitar su manipulación y se eliminan valores faltantes para asegurar la calidad del dataset. 

In [6]:
df = df.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
})

df = df.dropna()

Posteriormente, el índice se convierte a formato datetime y se ordena cronológicamente, lo cual es fundamental para el análisis de series de tiempo.

Después, se grafica el precio de cierre del activo utilizando Plotly, incorporando un range slider que permite explorar de forma interactiva la evolución del precio de la gasolina RBOB a lo largo del tiempo.

In [7]:
# cnvertir el índice a datetime y ordenar por fecha
df.index = pd.to_datetime(df.index)
df = df.sort_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df[("close", "RB=F")],
    mode="lines",
    name="Precio"
))

fig.update_layout(
    title="Precio de RBOB Gasoline (RB=F)",
    xaxis_title="Fecha",
    yaxis_title="Precio",
    template="plotly_white",
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show()

## Modelo FFNN

Se utiliza únicamente el precio de cierre como variable de entrada y se divide el dataset en conjuntos de entrenamiento y prueba (80/20).

In [8]:
# Usamos solo el precio de cierre
serie_tiempo = df['close'].dropna().values.reshape(-1, 1)

In [9]:
# Seleccionamos train y test (80/20)
train_size = int(len(serie_tiempo) * 0.8)
train_data = serie_tiempo[:train_size]
test_data = serie_tiempo[train_size:]

Aplicamos una normalización con MinMaxScaler, ajustándolo solo con los datos de entrenamiento para evitar fuga de información y transformando el conjunto de prueba con los mismos parámetros.

In [10]:
# Iniciamos el escalador
scaler = MinMaxScaler(feature_range=(0, 1))

# Aplicamos el escalador SOLO a los datos de entrenamiento
train_scaled = scaler.fit_transform(train_data)

# Ahora escalador con parametros del train, pero para transformar los datos de test
test_scaled = scaler.transform(test_data)

Para este modelo se construyen ventanas deslizantes de tamaño 10, donde cada conjunto de entradas (X) contiene los últimos 10 valores del precio y la salida (y) corresponde al siguiente valor. Esto permite adaptar la serie de tiempo a un formato supervisado adecuado para entrenar la red neuronal.

In [11]:
# Función para crear ventanas deslizantes
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size), 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

window_size = 10
X_train, y_train = crear_ventanas(train_scaled, window_size)
X_test, y_test = crear_ventanas(test_scaled, window_size)

#### Arquitectura del modelo

Se define un modelo FFNN compuesta por dos capas ocultas con funciones de activación ReLU, las cuales permiten capturar relaciones no lineales en los datos. La capa de entrada recibe como dimensión el tamaño de la ventana, mientras que la capa de salida cuenta con una sola neurona y activación lineal, adecuada para un problema de regresión.

El modelo se compila utilizando el optimizador Adam con una tasa de aprendizaje de 0.005 y la función de pérdida MSE (error cuadrático medio). Finalmente, se entrena durante 60 épocas con un tamaño de lote de 8, utilizando el conjunto de prueba como validación para monitorear el desempeño fuera de muestra.

In [12]:
# Iniciamos el modelo FFNN
model = Sequential([
    # Capa 1: input_dim DEBE coincidir con el window_size
    Dense(16, activation='relu', input_dim=window_size, name='Capa_Oculta_1'),

    # Capa 2: Extracción de características no lineales
    Dense(8, activation='relu', name='Capa_Oculta_2'),

    # Capa de Salida: 1 neurona Funcion de act LINEAL
    Dense(1, name='Salida_Pronostico')
])
model.summary()

# Usamos el optimizador Adam y MSE
optimizador = Adam(learning_rate=0.005)
model.compile(optimizer=optimizador, loss='mse')

history = model.fit(
    X_train, y_train,
    epochs=60,           # Cantidad de veces que verá el dataset completo
    batch_size=8,        # Actualiza los pesos cada 8 ventanas
    validation_data=(X_test, y_test), # Evaluamos en datos que no ha visto
    verbose=0            # 0 para no mostrar info
)


c:\Users\samys\Documents\Software\Anaconda\envs\MachineLearning\lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Capa_Oculta_1 (Dense)           │ (None, 16)             │           176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_Oculta_2 (Dense)           │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Salida_Pronostico (Dense)       │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

Una vez entrenado el modelo, se generan predicciones sobre el conjunto de prueba. Posteriormente, tanto las predicciones como los valores reales se transforman de nuevo a su escala original para facilitar su interpretación.

In [13]:
# Predicciones
preds = model.predict(X_test)

# Asegurar formato 2D
preds = preds.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

# Desescalar
preds = scaler.inverse_transform(preds)
y_test_real = scaler.inverse_transform(y_test)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


In [14]:
# Copia del dataframe
df_plot = df.copy()

# Asegurar índice datetime y ordenado
df_plot.index = pd.to_datetime(df_plot.index)
df_plot = df_plot.sort_index()

# Tomar la columna close real desde el MultiIndex
df_plot["real"] = df_plot[("close", "RB=F")]

# Crear columna de predicción
df_plot["pred"] = np.nan

# Insertar predicciones en el tramo final
df_plot.iloc[-len(preds):, df_plot.columns.get_loc("pred")] = preds.flatten()

# Crear gráfica interactiva
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot.index,
    y=df_plot["real"],
    mode="lines",
    name="Real"
))

fig.add_trace(go.Scatter(
    x=df_plot.index,
    y=df_plot["pred"],
    mode="lines",
    name="Predicción"
))

fig.update_layout(
    title="Precio Real vs Predicción - RBOB",
    xaxis_title="Fecha",
    yaxis_title="Precio",
    template="plotly_white",
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show()

In [15]:
# Métricas
mae = mean_absolute_error(y_test_real, preds)
rmse = np.sqrt(mean_squared_error(y_test_real, preds))

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

MAE: 0.036
RMSE: 0.054


In [16]:
real = np.sign(np.diff(y_test_real.flatten()))
pred = np.sign(np.diff(preds.flatten()))

DA = np.mean(real == pred)

print("Directional Accuracy:", DA)

Directional Accuracy: 0.4685598377281947


El desempeño del modelo se evalúa mediante métricas de error y precisión direccional. Se obtiene un MAE de 0.040 y un RMSE de 0.059, lo que indica que, en promedio, las predicciones se mantienen relativamente cercanas a los valores reales.

Sin embargo, la precisión direccional (Directional Accuracy) es de 0.484, lo cual se encuentra por debajo del umbral deseado del 55%. Esto sugiere que, aunque el modelo logra aproximar los niveles de precio, presenta limitaciones para predecir correctamente la dirección del movimiento del mercado, lo que reduce su utilidad en la toma de decisiones de inversión.

## Segundo modelo FNN

En este segundo modelo, se incrementa el tamaño de la ventana deslizante a 20 periodos. Esto implica que el modelo ahora utiliza más información histórica para realizar cada predicción, lo que puede ayudar a capturar patrones más complejos y de mayor plazo en la serie de tiempo.

Al igual que en el caso anterior, se generan los conjuntos de entrada (X) y salida (y) tanto para entrenamiento como para prueba, adaptando la serie a un formato supervisado adecuado para la red neuronal.

In [17]:
# Función para crear ventanas deslizantes
def crear_ventanas(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size), 0])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

window_size = 20
X_train, y_train = crear_ventanas(train_scaled, window_size)
X_test, y_test = crear_ventanas(test_scaled, window_size)

#### Arquitectura

En este segundo modelo se incrementa la complejidad de la red neuronal, incorporando más capas y un mayor número de neuronas (128, 64 y 32), lo que permite capturar patrones más profundos en los datos. Además, se incluyen capas de Dropout para reducir el sobreajuste, desactivando aleatoriamente ciertas neuronas durante el entrenamiento.

La capa de salida utiliza una activación sigmoide, mientras que el modelo se compila nuevamente con el optimizador Adam, pero con una tasa de aprendizaje menor (0.001) para lograr un entrenamiento más estable. Finalmente, el modelo se entrena durante 150 épocas con un tamaño de lote de 32, evaluando su desempeño con datos de prueba en cada iteración.

In [18]:
model_two = Sequential([
   Dense(128, activation='relu', input_shape=(window_size,)),
   Dropout(0.2),
   Dense(64, activation='relu'),
   Dropout(0.2),
   Dense(32, activation='relu'),
   Dropout(0.1),
   Dense(1, activation='sigmoid')
])

model_two.summary()

optimizador = Adam(learning_rate=0.001)
model_two.compile(optimizer=optimizador, loss='mse')

history = model_two.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

c:\Users\samys\Documents\Software\Anaconda\envs\MachineLearning\lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,057 (51.00 KB)

 Trainable params: 13,057 (51.00 KB)

 Non-trainable params: 0 (0.00 B)

### Predicciones para el 13 de abril 

In [19]:
# Fechas objetivo: 13-17 Abril 2026 (Lunes a Viernes)
target_dates = pd.date_range(start='2026-04-13', periods=5, freq='B')
print(f"Fechas objetivo para predicción:")
for date in target_dates:
    print(f"  - {date.strftime('%Y-%m-%d')} ({date.strftime('%A')})")

# Usar últimos 20 días del dataset como entrada para predicción
last_sequence = train_scaled[-window_size:].reshape(1, window_size)
predictions_two = []
current_sequence = last_sequence.copy()

print(f"\nGenerando predicciones secuenciales")
for i in range(5):  # 5 días
    next_pred = model_two.predict(current_sequence, verbose=0)[0, 0]
    predictions_two.append(next_pred)
    
    # Actualizar secuencia
    current_sequence = np.roll(current_sequence, -1, axis=1)
    current_sequence[0, -1] = next_pred

# Desnormalizar predicciones
predictions_two_actual = scaler.inverse_transform(np.array(predictions_two).reshape(-1, 1)).flatten()

# Crear DataFrame de predicciones
df_forecast_two = pd.DataFrame({
    'date': target_dates,
    'day_of_week': target_dates.strftime('%A'),
    'predicted_close': predictions_two_actual
})

print("\n Predicciones 13-17 Abril")
print(df_forecast_two.to_string(index=False))

# Estadísticas de predicción
print(f"\nEstadísticas de Predicción:")
print(f"  Precio promedio predicho: ${predictions_two_actual.mean():.2f}")
print(f"  Mínimo predicho: ${predictions_two_actual.min():.2f}")
print(f"  Máximo predicho: ${predictions_two_actual.max():.2f}")
print(f"  Volatilidad (std): ${predictions_two_actual.std():.4f}")

Fechas objetivo para predicción:
  - 2026-04-13 (Monday)
  - 2026-04-14 (Tuesday)
  - 2026-04-15 (Wednesday)
  - 2026-04-16 (Thursday)
  - 2026-04-17 (Friday)

Generando predicciones secuenciales

 Predicciones 13-17 Abril
      date day_of_week  predicted_close
2026-04-13      Monday         2.627912
2026-04-14     Tuesday         2.548293
2026-04-15   Wednesday         2.485365
2026-04-16    Thursday         2.453111
2026-04-17      Friday         2.417912

Estadísticas de Predicción:
  Precio promedio predicho: $2.51
  Mínimo predicho: $2.42
  Máximo predicho: $2.63
  Volatilidad (std): $0.0743


El modelo genera las predicciones para la semana objetivo obteniendo una tendencia a la baja en el precio de la gasolina RBOB a lo largo de los cinco días. Los valores muestran una disminución progresiva desde aproximadamente 2.68 hasta 2.53.

El precio promedio predicho es de 2.60, con una variación acotada entre 2.53 y 2.68, y una volatilidad relativamente baja (desviación estándar de 0.0507). Esto sugiere un escenario de ligera tendencia bajista con movimientos moderados durante el periodo analizado.

In [20]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df[("close", "RB=F")],
    mode="lines",
    name="Precio histórico"
))

fig.add_trace(go.Scatter(
    x=df_forecast_two["date"],
    y=df_forecast_two["predicted_close"],
    mode="lines+markers",
    name="Predicción 13-17 Abr",
    line=dict(color="red", width=2, dash="dash")
))

fig.update_layout(
    title="Predicción de cierre para 13-17 Abril 2026",
    xaxis_title="Fecha",
    yaxis_title="Precio",
    template="plotly_white",
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show()

---------------------------------------------------------

## Modelo Garch

Para el modelado GARCH, se calculan los retornos logarítmicos diarios del precio de cierre, expresados en porcentaje. Esta transformación es fundamental, ya que los modelos de volatilidad trabajan sobre retornos en lugar de precios, permitiendo analizar de forma más adecuada la variabilidad del activo.


In [21]:
# Calcular retornos
returns = 100 * np.log(df['close'] / df['close'].shift(1)).dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines', name='Retornos'))
fig.update_layout(title=f'Retornos Diarios',
                  yaxis_title='Retornos (%)')
fig.show()

Se analizan los retornos al cuadrado con funciones de autocorrelación (ACF) y autocorrelación parcial (PACF) para identificar la presencia de heterocedasticidad condicional. Esto nos ayuda, ya que los modelos GARCH se justifican cuando existe dependencia en la varianza a lo largo del tiempo.

In [22]:
# Retornos al cuadrado
sq_returns = returns**2

# Calcular ACF y PACF
lag_acf = acf(sq_returns, nlags=20)
lag_pacf = pacf(sq_returns, nlags=20, method='ols')

# Graficar ACF y PACF con Plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('ACF de Retornos al Cuadrado', 'PACF de Retornos al Cuadrado'))

# Añadir barras de ACF
fig.add_trace(go.Bar(x=np.arange(len(lag_acf)), y=lag_acf, name='ACF'), row=1, col=1)
# Añadir barras de PACF
fig.add_trace(go.Bar(x=np.arange(len(lag_pacf)), y=lag_pacf, name='PACF'), row=1, col=2)

# Líneas de significancia (aprox 1.96 / sqrt(N))
sig_level = 1.96 / np.sqrt(len(returns))
for i in [1, 2]:
    fig.add_hline(y=sig_level, line_dash="dash", line_color="red", row=1, col=i)
    fig.add_hline(y=-sig_level, line_dash="dash", line_color="red", row=1, col=i)

fig.update_layout(title='Análisis de Dependencia de Varianza', showlegend=False)
fig.show()

En las gráficas se observan varios rezagos que superan los niveles de significancia, lo que indica que la volatilidad no es constante y presenta agrupamiento. Este comportamiento confirma la presencia de efectos ARCH.

Se ajusta un modelo GARCH(1,1) sobre los retornos, asumiendo una distribución t para capturar mejor colas pesadas típicas de los datos financieros. 

A partir de esta volatilidad, se calcula el Value at Risk (VaR) al 5%, utilizando el cuantil empírico de los residuales estandarizados. El VaR representa la pérdida máxima esperada bajo condiciones normales con un 95% de confianza.

Finalmente, se grafica la serie de retornos junto con el VaR estimado, lo que permite visualizar en qué momentos las pérdidas reales superan el umbral de riesgo, evaluando así la capacidad del modelo para capturar eventos extremos.

In [26]:
# Ajuste del modelo GARCH(1,1)
am = arch_model(returns, vol='Garch', p=1, q=1, dist='t')
res = am.fit(disp='off')


# Extraemos la volatilidad condicional modelada (Varianza que predijimos)
cond_vol = res.conditional_volatility

# Cálculo del Value at Risk (VaR) Histórico Condicional (GARCH) al 5%
# Cuantil empírico al 5% de los residuales estandarizados
q_5 = res.std_resid.quantile(0.05)

# VaR = Media Condicional + (Cuantil * Volatilidad Condicional)
# GARCH por defecto asume media constante, la extraemos de los parámetros
mean = res.params['mu']
VaR_5 = mean + cond_vol * q_5

# 5. Graficar los Retornos vs el VaR predictivo
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines',
                         name='Retornos Reales', opacity=0.6))
fig.add_trace(go.Scatter(x=VaR_5.index, y=VaR_5, mode='lines',
                         name='VaR 5% (GARCH)', line=dict(color='red')))

fig.update_layout(title='Backtesting de Riesgo: Retornos de XOM vs GARCH(1,1) VaR al 5%',
                  yaxis_title='Retornos (%)')
fig.show()

En la gráfica de backtesting, se c observa que la mayoría de las caídas extremas se mantienen por encima del umbral estimado, y solo en pocos casos los retornos lo sobrepasan, lo cual es consistente con el nivel de confianza del 95%.

In [27]:
print(res.summary())

                        Constant Mean - GARCH Model Results                         
Dep. Variable:                         RB=F   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -5562.39
Distribution:      Standardized Student's t   AIC:                           11134.8
Method:                  Maximum Likelihood   BIC:                           11163.9
                                              No. Observations:                 2515
Date:                      Sun, Apr 12 2026   Df Residuals:                     2514
Time:                              17:02:36   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu        

El modelo GARCH(1,1) ajustado muestra parámetros estadísticamente significativos, destacando un alto valor de persistencia en la volatilidad (α + β ≈ 0.95), lo que indica que los choques en la volatilidad tienden a mantenerse en el tiempo. 

Estos resultados sugieren que el modelo GARCH logra capturar adecuadamente la dinámica de la volatilidad y proporciona una estimación razonable del riesgo, cumpliendo con el objetivo de limitar las pérdidas extremas esperadas en el tiempo.


In [28]:
# Número de días a predecir
horizon = 5

# Generar pronóstico de volatilidad
forecast = res.forecast(horizon=horizon)

# Extraer la varianza pronosticada para los próximos días
forecast_var = forecast.variance.values[-1, :]
forecast_vol = np.sqrt(forecast_var)

# Media estimada del modelo
mean = res.params['mu']

# Cuantil al 5% de los residuales estandarizados
q_5 = res.std_resid.quantile(0.05)

# Calcular VaR para los próximos días
VaR_forecast = mean + forecast_vol * q_5

# Fechas objetivo
target_dates = pd.date_range(start='2026-04-13', periods=5, freq='B')

# Crear DataFrame con resultados
df_garch_forecast = pd.DataFrame({
    'date': target_dates,
    'day_of_week': target_dates.strftime('%A'),
    'volatility_forecast': forecast_vol,
    'VaR_95': VaR_forecast
})

print("\nPredicciones GARCH (Volatilidad y VaR 95%)")
print(df_garch_forecast.to_string(index=False))

# Estadísticas
print("\nEstadísticas de Riesgo:")
print(f"  Volatilidad promedio: {forecast_vol.mean():.4f}")
print(f"  VaR promedio: {VaR_forecast.mean():.4f}")
print(f"  Peor VaR (más negativo): {VaR_forecast.min():.4f}")


Predicciones GARCH (Volatilidad y VaR 95%)
      date day_of_week  volatility_forecast    VaR_95
2026-04-13      Monday             3.719997 -6.122702
2026-04-14     Tuesday             3.663468 -6.027731
2026-04-15   Wednesday             3.609195 -5.936550
2026-04-16    Thursday             3.557109 -5.849042
2026-04-17      Friday             3.507139 -5.765091

Estadísticas de Riesgo:
  Volatilidad promedio: 3.6114
  VaR promedio: -5.9402
  Peor VaR (más negativo): -6.1227


El modelo GARCH(1,1) muestra una ligera disminución en la volatilidad durante la semana, indicando una reducción en la incertidumbre del mercado.

El VaR al 95% se mantiene cercano a -6%, con el peor escenario el 13 de abril (-6.12%) y una leve mejora en los días siguientes. En promedio, el riesgo estimado es de -5.94%, lo que sugiere un nivel de riesgo relativamente estable.